In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo
from pathlib import Path

import pandas as pd
from xbbg import blp, Backend


In [2]:

# ============================================================
# Settings
# ============================================================

TZ = "Asia/Tokyo"
LOOKBACK_YEARS = 5

OUTPUT_DIR = Path(r"C:\Data\DCAM")

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# Bloomberg Tickers
#
# Principles:
# - Keep canonical_id stable where possible
# - Store raw daily levels
# - Calculate returns / volatility / correlation later
# - Use total-return data where economically appropriate
# ============================================================

TICKERS = {

    # --------------------------------------------------------
    # US / Japan equity futures
    # --------------------------------------------------------
    "ES": "ES1 A:00_0_R COMB Index",
    "NQ": "NQ1 A:00_0_R COMB Index",
    "RTY": "RTY1 A:00_0_R COMB Index",
    "TOPIX": "TP1 A:00_0_R COMB Index",

    # --------------------------------------------------------
    # J-REIT
    # TSE REIT Total Return Index
    # --------------------------------------------------------
    "JREIT": "TPXDREIT Index",

    # --------------------------------------------------------
    # European equity futures
    # --------------------------------------------------------
    "EURO": "VG1 A:00_0_R COMB Index",
    "DAX": "GX1 A:00_0_R COMB Index",

    # --------------------------------------------------------
    # Asia equity futures
    # --------------------------------------------------------
    "TAIWAN": "TWT1 A:00_0_R COMB Index",
    "KOSPI": "KM1 A:00_0_R COMB Index",
    "CHINA_A50": "XU1 A:00_0_R COMB Index",
    "HSI": "HI1 A:00_0_R COMB Index",
    "NIFTY50": "JGS1 A:00_0_R COMB Index",

    # --------------------------------------------------------
    # Volatility
    # --------------------------------------------------------
    "VIX": "VIX Index",
    "VIX_F1": "UX1 Index",
    "MOVE": "MOVE Index",

    # --------------------------------------------------------
    # US Treasury futures
    # WN = Ultra T-Bond futures
    # Keep US30Y canonical_id for compatibility with ICAM
    # --------------------------------------------------------
    "US2Y": "TU1 A:00_0_R COMB Comdty",
    "US5Y": "FV1 A:00_0_R COMB Comdty",
    "US10Y": "TY1 A:00_0_R COMB Comdty",
    "US30Y": "WN1 A:00_0_R COMB Comdty",

    # --------------------------------------------------------
    # US nominal Treasury yields
    # Unit = %
    # --------------------------------------------------------
    "US2Y_YIELD": "USGG2YR Index",
    "US5Y_YIELD": "USGG5YR Index",
    "US10Y_YIELD": "USGG10YR Index",
    "US30Y_YIELD": "USGG30YR Index",

    # --------------------------------------------------------
    # US real yields
    # Unit = %
    # --------------------------------------------------------
    "US5Y_REAL": "USGGT05Y Index",
    "US10Y_REAL": "USGGT10Y Index",

    # --------------------------------------------------------
    # US breakeven inflation
    # Unit = %
    # --------------------------------------------------------
    "US5Y_BE": "USGGBE05 Index",
    "US10Y_BE": "USGGBE10 Index",

    # --------------------------------------------------------
    # US credit spreads
    #
    # Raw Bloomberg values:
    #   IG 0.77 = 0.77% = 77bp
    #   HY 2.65 = 2.65% = 265bp
    #
    # Store raw level in PCT.
    # Convert changes to bp during analysis.
    # --------------------------------------------------------
    "US_IG_OAS": "LUACOAS Index",
    "US_HY_OAS": "LF98OAS Index",

    # --------------------------------------------------------
    # European rates futures
    # --------------------------------------------------------
    "SCHATZ": "DU1 A:00_0_R COMB Comdty",
    "BOBL": "OE1 A:00_0_R COMB Comdty",
    "BUND": "RX1 A:00_0_R COMB Comdty",
    "BUXL": "UB1 A:00_0_R COMB Comdty",
    "BTP": "IK1 A:00_0_R COMB Comdty",

    # --------------------------------------------------------
    # Japanese rates futures
    # --------------------------------------------------------
    "JGB": "JB1 A:00_0_R COMB Comdty",

    # --------------------------------------------------------
    # FX spot
    # --------------------------------------------------------
    "USDJPY": "USDJPY BGN Curncy",
    "EURUSD": "EURUSD BGN Curncy",
    "GBPUSD": "GBPUSD BGN Curncy",
    "AUDUSD": "AUDUSD BGN Curncy",
    "USDCNH": "USDCNH BGN Curncy",
    "DXY": "DXY Index",

    # --------------------------------------------------------
    # FX option ATM implied volatility
    # --------------------------------------------------------
    "USDJPY_VOL_1M": "USDJPYV1M Curncy",
    "USDJPY_VOL_3M": "USDJPYV3M Curncy",
    "EURUSD_VOL_1M": "EURUSDV1M Curncy",
    "EURUSD_VOL_3M": "EURUSDV3M Curncy",

    # --------------------------------------------------------
    # FX option 25-delta risk reversal
    # --------------------------------------------------------
    "USDJPY_RR25_1M": "USDJPY25R1M Curncy",
    "USDJPY_RR25_3M": "USDJPY25R3M Curncy",
    "EURUSD_RR25_1M": "EURUSD25R1M Curncy",
    "EURUSD_RR25_3M": "EURUSD25R3M Curncy",

    # --------------------------------------------------------
    # Energy
    # --------------------------------------------------------
    "WTI": "CL1 A:00_0_R COMB Comdty",
    "BRENT": "CO1 Comdty",

    # --------------------------------------------------------
    # Metals
    # --------------------------------------------------------
    "GOLD": "GC1 A:00_0_R COMB Comdty",
    "SILVER": "SI1 A:00_0_R COMB Comdty",
    "COPPER": "HG1 A:00_0_R COMB Comdty",

    # --------------------------------------------------------
    # Agriculture
    # --------------------------------------------------------
    "CORN": "C 1 A:00_0_R COMB Comdty",
    "WHEAT": "W 1 A:00_0_R COMB Comdty",
    "SOYBEAN": "S 1 A:00_0_R COMB Comdty",

    # --------------------------------------------------------
    # US equity sectors
    #
    # ETFs are used as sector proxies.
    # Bloomberg field will be:
    # TOT_RETURN_INDEX_GROSS_DVDS
    #
    # Therefore distributions are included.
    # --------------------------------------------------------
    "US_TECH": "XLK US Equity",
    "US_FINANCIALS": "XLF US Equity",
    "US_HEALTHCARE": "XLV US Equity",
    "US_CONSUMER_DISC": "XLY US Equity",
    "US_COMM_SERVICES": "XLC US Equity",
    "US_INDUSTRIALS": "XLI US Equity",
    "US_CONSUMER_STAPLES": "XLP US Equity",
    "US_ENERGY": "XLE US Equity",
    "US_UTILITIES": "XLU US Equity",
    "US_MATERIALS": "XLB US Equity",
    "US_REAL_ESTATE": "XLRE US Equity",
}


# ============================================================
# Japan Equity Sectors
#
# Add TOPIX-17 after confirming exact Bloomberg identifiers.
# ============================================================

JAPAN_SECTOR_TICKERS = {
    # "JP_FOODS": "...",
    # "JP_ENERGY": "...",
    # "JP_CONSTRUCTION_MATERIALS": "...",
    # "JP_RAW_MATERIALS_CHEMICALS": "...",
    # "JP_PHARMA": "...",
    # "JP_AUTOS_TRANSPORT_EQUIP": "...",
    # "JP_STEEL_NONFERROUS": "...",
    # "JP_MACHINERY": "...",
    # "JP_ELECTRIC_PRECISION": "...",
    # "JP_IT_SERVICES": "...",
    # "JP_ELECTRIC_POWER_GAS": "...",
    # "JP_TRANSPORT_LOGISTICS": "...",
    # "JP_COMMERCIAL_WHOLESALE": "...",
    # "JP_RETAIL": "...",
    # "JP_BANKS": "...",
    # "JP_FINANCIALS_EX_BANKS": "...",
    # "JP_REAL_ESTATE": "...",
}

TICKERS.update(
    JAPAN_SECTOR_TICKERS
)


# ============================================================
# Series Classification
# ============================================================

YIELD_IDS = {
    "US2Y_YIELD",
    "US5Y_YIELD",
    "US10Y_YIELD",
    "US30Y_YIELD",
}

REAL_YIELD_IDS = {
    "US5Y_REAL",
    "US10Y_REAL",
}

BREAKEVEN_IDS = {
    "US5Y_BE",
    "US10Y_BE",
}

CREDIT_SPREAD_IDS = {
    "US_IG_OAS",
    "US_HY_OAS",
}

VOLATILITY_IDS = {
    "VIX",
    "VIX_F1",
    "MOVE",
}

FX_SPOT_IDS = {
    "USDJPY",
    "EURUSD",
    "GBPUSD",
    "AUDUSD",
    "USDCNH",
    "DXY",
}

FX_VOL_IDS = {
    "USDJPY_VOL_1M",
    "USDJPY_VOL_3M",
    "EURUSD_VOL_1M",
    "EURUSD_VOL_3M",
}

FX_RR_IDS = {
    "USDJPY_RR25_1M",
    "USDJPY_RR25_3M",
    "EURUSD_RR25_1M",
    "EURUSD_RR25_3M",
}

US_SECTOR_IDS = {
    "US_TECH",
    "US_FINANCIALS",
    "US_HEALTHCARE",
    "US_CONSUMER_DISC",
    "US_COMM_SERVICES",
    "US_INDUSTRIALS",
    "US_CONSUMER_STAPLES",
    "US_ENERGY",
    "US_UTILITIES",
    "US_MATERIALS",
    "US_REAL_ESTATE",
}


# ------------------------------------------------------------
# Series that economically represent total return indices
#
# JREIT:
#   ticker itself is already a total-return index
#
# US sectors:
#   Bloomberg total-return field is requested on the ETF
# ------------------------------------------------------------

TOTAL_RETURN_INDEX_IDS = (
    US_SECTOR_IDS
    | {
        "JREIT",
    }
)


# ------------------------------------------------------------
# Series requiring a Bloomberg field other than PX_LAST
# ------------------------------------------------------------

TOTAL_RETURN_FIELD_IDS = (
    US_SECTOR_IDS
)


# ============================================================
# Bloomberg Field
# ============================================================

def get_bloomberg_field(canonical_id):

    if canonical_id in TOTAL_RETURN_FIELD_IDS:
        return "TOT_RETURN_INDEX_GROSS_DVDS"

    return "PX_LAST"


# ============================================================
# Metadata Functions
# ============================================================

def get_series_type(canonical_id):

    if canonical_id in YIELD_IDS:
        return "NOMINAL_YIELD"

    if canonical_id in REAL_YIELD_IDS:
        return "REAL_YIELD"

    if canonical_id in BREAKEVEN_IDS:
        return "BREAKEVEN"

    if canonical_id in CREDIT_SPREAD_IDS:
        return "CREDIT_SPREAD"

    if canonical_id in VOLATILITY_IDS:
        return "VOLATILITY"

    if canonical_id in FX_VOL_IDS:
        return "IMPLIED_VOL"

    if canonical_id in FX_RR_IDS:
        return "RISK_REVERSAL"

    if canonical_id in TOTAL_RETURN_INDEX_IDS:
        return "TOTAL_RETURN_INDEX"

    return "PRICE"


def get_unit(canonical_id):

    if (
        canonical_id in YIELD_IDS
        or canonical_id in REAL_YIELD_IDS
        or canonical_id in BREAKEVEN_IDS
        or canonical_id in CREDIT_SPREAD_IDS
    ):
        return "PCT"

    if canonical_id in FX_VOL_IDS:
        return "VOL_PCT"

    if canonical_id in FX_RR_IDS:
        return "VOL_POINT"

    if canonical_id in TOTAL_RETURN_INDEX_IDS:
        return "INDEX"

    return "PRICE"


def get_asset_class(canonical_id):

    if canonical_id in US_SECTOR_IDS:
        return "EQUITY_SECTOR"

    if canonical_id.startswith("JP_"):
        return "EQUITY_SECTOR"

    if canonical_id in {
        "ES",
        "NQ",
        "RTY",
        "TOPIX",
        "EURO",
        "DAX",
        "TAIWAN",
        "KOSPI",
        "CHINA_A50",
        "HSI",
        "NIFTY50",
    }:
        return "EQUITY"

    if canonical_id == "JREIT":
        return "REIT"

    if canonical_id in VOLATILITY_IDS:
        return "VOLATILITY"

    if (
        canonical_id in YIELD_IDS
        or canonical_id in REAL_YIELD_IDS
        or canonical_id in BREAKEVEN_IDS
    ):
        return "RATES"

    if canonical_id in CREDIT_SPREAD_IDS:
        return "CREDIT"

    if canonical_id in {
        "US2Y",
        "US5Y",
        "US10Y",
        "US30Y",
        "SCHATZ",
        "BOBL",
        "BUND",
        "BUXL",
        "BTP",
        "JGB",
    }:
        return "RATES"

    if canonical_id in FX_SPOT_IDS:
        return "FX"

    if (
        canonical_id in FX_VOL_IDS
        or canonical_id in FX_RR_IDS
    ):
        return "FX_OPTION"

    if canonical_id in {
        "WTI",
        "BRENT",
    }:
        return "ENERGY"

    if canonical_id in {
        "GOLD",
        "SILVER",
        "COPPER",
    }:
        return "METALS"

    if canonical_id in {
        "CORN",
        "WHEAT",
        "SOYBEAN",
    }:
        return "AGRICULTURE"

    return "OTHER"


def get_region(canonical_id):

    # --------------------------------------------------------
    # FX / FX options are treated as global cross-asset series
    # --------------------------------------------------------

    if (
        canonical_id in FX_SPOT_IDS
        or canonical_id in FX_VOL_IDS
        or canonical_id in FX_RR_IDS
    ):
        return "GLOBAL"

    # --------------------------------------------------------
    # US
    # --------------------------------------------------------

    if (
        canonical_id.startswith("US")
        or canonical_id in {
            "ES",
            "NQ",
            "RTY",
            "VIX",
            "VIX_F1",
            "MOVE",
        }
    ):
        return "US"

    # --------------------------------------------------------
    # Japan
    # --------------------------------------------------------

    if (
        canonical_id.startswith("JP_")
        or canonical_id in {
            "TOPIX",
            "JREIT",
            "JGB",
        }
    ):
        return "JAPAN"

    # --------------------------------------------------------
    # Europe
    # --------------------------------------------------------

    if canonical_id in {
        "EURO",
        "DAX",
        "SCHATZ",
        "BOBL",
        "BUND",
        "BUXL",
        "BTP",
    }:
        return "EUROPE"

    # --------------------------------------------------------
    # Asia
    # --------------------------------------------------------

    if canonical_id in {
        "TAIWAN",
        "KOSPI",
        "CHINA_A50",
        "HSI",
        "NIFTY50",
    }:
        return "ASIA"

    return "GLOBAL"


# ============================================================
# Normalize BDH Output
#
# Supports:
# - newer long-form xbbg output
# - traditional index/MultiIndex pandas output
# ============================================================

def normalize_bdh_output(
    df,
    requested_field,
):

    if df is None or df.empty:

        return pd.DataFrame(
            columns=[
                "date",
                "close",
            ]
        )

    df = df.copy()

    lower_columns = {
        str(col).lower(): col
        for col in df.columns
    }

    # --------------------------------------------------------
    # Case 1:
    # Long-format output
    #
    # ticker | field | date | value
    # --------------------------------------------------------

    if (
        "date" in lower_columns
        and "value" in lower_columns
    ):

        date_col = lower_columns["date"]
        value_col = lower_columns["value"]

        if "field" in lower_columns:

            field_col = lower_columns["field"]

            df = df.loc[
                df[field_col]
                .astype(str)
                .str.upper()
                .eq(
                    requested_field.upper()
                )
            ]

        result = df[
            [
                date_col,
                value_col,
            ]
        ].copy()

        result.columns = [
            "date",
            "close",
        ]

    # --------------------------------------------------------
    # Case 2:
    # Traditional pandas / MultiIndex output
    # --------------------------------------------------------

    else:

        if df.shape[1] == 0:

            return pd.DataFrame(
                columns=[
                    "date",
                    "close",
                ]
            )

        value_series = df.iloc[:, 0]

        result = (
            value_series
            .rename("close")
            .reset_index()
        )

        result = result.rename(
            columns={
                result.columns[0]: "date",
            }
        )

    # --------------------------------------------------------
    # Data types
    # --------------------------------------------------------

    result["date"] = (
        pd.to_datetime(
            result["date"],
            errors="coerce",
        )
        .dt.date
    )

    result["close"] = pd.to_numeric(
        result["close"],
        errors="coerce",
    )

    result = (
        result
        .dropna(
            subset=[
                "date",
                "close",
            ]
        )
        .sort_values(
            "date"
        )
        .reset_index(
            drop=True
        )
    )

    return result


# ============================================================
# Function: Get Daily Bloomberg History
# ============================================================

def get_daily_history(
    tickers,
    lookback_years=5,
    tz="Asia/Tokyo",
):

    """
    Retrieve Bloomberg daily historical data.

    Default:
        5 calendar years.

    Only completed daily observations are requested:
        end_date = previous calendar day.

    Output:
        date
        canonical_id
        asset_class
        region
        series_type
        unit
        close
        source_ticker
        source_field

    Notes:
        - No forward filling.
        - No return calculation.
        - No volatility calculation.
        - Preserve raw Bloomberg observations.
        - Different markets may have different latest dates.
    """

    # --------------------------------------------------------
    # Dates
    # --------------------------------------------------------

    today = pd.Timestamp(
        datetime.now(
            ZoneInfo(tz)
        ).date()
    )

    # Do not request today's potentially incomplete daily bar.
    end_date_ts = (
        today
        - pd.Timedelta(
            days=1
        )
    )

    end_date = (
        end_date_ts.date()
    )

    start_date = (
        end_date_ts
        - pd.DateOffset(
            years=lookback_years
        )
    ).date()

    print("=" * 80)
    print("DCAM Bloomberg Daily Download")
    print("=" * 80)

    print(
        f"Timezone    : {tz}"
    )

    print(
        f"Start date  : {start_date}"
    )

    print(
        f"End date    : {end_date}"
    )

    print(
        f"History     : {lookback_years} years"
    )

    print(
        f"Instruments : {len(tickers)}"
    )

    print()

    # --------------------------------------------------------
    # Containers
    # --------------------------------------------------------

    dfs = []
    errors = []
    no_data = []

    # --------------------------------------------------------
    # Bloomberg requests
    # --------------------------------------------------------

    for canonical_id, ticker in tickers.items():

        field = get_bloomberg_field(
            canonical_id
        )

        try:

            raw = blp.bdh(
                ticker,
                field,
                start_date=start_date,
                end_date=end_date,
                Per="D",
                backend=Backend.PANDAS,
            )

            df = normalize_bdh_output(
                raw,
                requested_field=field,
            )

            # ------------------------------------------------
            # No Data
            # ------------------------------------------------

            if df.empty:

                no_data.append(
                    {
                        "canonical_id": canonical_id,
                        "ticker": ticker,
                        "field": field,
                    }
                )

                print(
                    f"NO DATA | "
                    f"{canonical_id:<24} | "
                    f"{field:<30} | "
                    f"{ticker}"
                )

                continue

            # ------------------------------------------------
            # Metadata
            # ------------------------------------------------

            df["canonical_id"] = (
                canonical_id
            )

            df["asset_class"] = (
                get_asset_class(
                    canonical_id
                )
            )

            df["region"] = (
                get_region(
                    canonical_id
                )
            )

            df["series_type"] = (
                get_series_type(
                    canonical_id
                )
            )

            df["unit"] = (
                get_unit(
                    canonical_id
                )
            )

            df["source_ticker"] = (
                ticker
            )

            df["source_field"] = (
                field
            )

            dfs.append(
                df
            )

            print(
                f"OK      | "
                f"{canonical_id:<24} | "
                f"{len(df):>5,} rows | "
                f"{df['date'].min()} -> "
                f"{df['date'].max()} | "
                f"{field}"
            )

        except Exception as e:

            errors.append(
                {
                    "canonical_id": canonical_id,
                    "ticker": ticker,
                    "field": field,
                    "error": str(e),
                }
            )

            print(
                f"ERROR   | "
                f"{canonical_id:<24} | "
                f"{field:<30} | "
                f"{ticker} | "
                f"{e}"
            )

    # --------------------------------------------------------
    # Logs
    # --------------------------------------------------------

    error_df = pd.DataFrame(
        errors
    )

    no_data_df = pd.DataFrame(
        no_data
    )

    # --------------------------------------------------------
    # No successful requests
    # --------------------------------------------------------

    if not dfs:

        return (
            pd.DataFrame(),
            error_df,
            no_data_df,
        )

    # --------------------------------------------------------
    # Combine
    # --------------------------------------------------------

    daily = pd.concat(
        dfs,
        ignore_index=True,
    )

    daily = (
        daily
        .sort_values(
            [
                "date",
                "canonical_id",
            ]
        )
        .reset_index(
            drop=True
        )
    )

    # --------------------------------------------------------
    # Column order
    # --------------------------------------------------------

    daily = daily[
        [
            "date",
            "canonical_id",
            "asset_class",
            "region",
            "series_type",
            "unit",
            "close",
            "source_ticker",
            "source_field",
        ]
    ]

    return (
        daily,
        error_df,
        no_data_df,
    )


# ============================================================
# Run
# ============================================================

daily_5y, errors, no_data = (
    get_daily_history(
        tickers=TICKERS,
        lookback_years=LOOKBACK_YEARS,
        tz=TZ,
    )
)


# ============================================================
# Summary
# ============================================================

print()
print("=" * 80)
print("DCAM RAW 5-YEAR DAILY DATA SUMMARY")
print("=" * 80)

print(
    f"Total rows  : "
    f"{len(daily_5y):,}"
)


if daily_5y.empty:

    print(
        "No Bloomberg data was returned."
    )

else:

    instrument_count = (
        daily_5y[
            "canonical_id"
        ]
        .nunique()
    )

    print(
        f"Instruments : "
        f"{instrument_count}"
    )

    print(
        f"First date  : "
        f"{daily_5y['date'].min()}"
    )

    print(
        f"Last date   : "
        f"{daily_5y['date'].max()}"
    )

    summary = (
        daily_5y
        .groupby(
            [
                "canonical_id",
                "asset_class",
                "region",
                "series_type",
                "unit",
                "source_field",
            ]
        )
        .agg(
            rows=(
                "date",
                "size",
            ),
            first_date=(
                "date",
                "min",
            ),
            last_date=(
                "date",
                "max",
            ),
            first_value=(
                "close",
                "first",
            ),
            last_value=(
                "close",
                "last",
            ),
        )
        .sort_index()
    )

    print()
    print(
        "Rows by instrument:"
    )

    display(
        summary
    )


# ============================================================
# Data Quality Check
# ============================================================

if not daily_5y.empty:

    duplicate_count = (
        daily_5y
        .duplicated(
            subset=[
                "canonical_id",
                "date",
            ]
        )
        .sum()
    )

    missing_close_count = (
        daily_5y[
            "close"
        ]
        .isna()
        .sum()
    )

    # --------------------------------------------------------
    # Only positive-level series.
    #
    # Do NOT apply to:
    # - yields
    # - risk reversals
    # because negative values may be valid.
    # --------------------------------------------------------

    positive_level_types = {
        "PRICE",
        "TOTAL_RETURN_INDEX",
        "VOLATILITY",
        "IMPLIED_VOL",
    }

    invalid_positive_level_count = (
        daily_5y.loc[
            daily_5y[
                "series_type"
            ].isin(
                positive_level_types
            ),
            "close",
        ]
        .le(0)
        .sum()
    )

    print()
    print("=" * 80)
    print("DATA QUALITY")
    print("=" * 80)

    print(
        f"Duplicate canonical_id/date : "
        f"{duplicate_count:,}"
    )

    print(
        f"Missing close               : "
        f"{missing_close_count:,}"
    )

    print(
        f"Invalid positive levels <=0 : "
        f"{invalid_positive_level_count:,}"
    )


# ============================================================
# Latest Observation Check
#
# Do not forward-fill raw data.
# Different markets have different holidays/publication lags.
# ============================================================

if not daily_5y.empty:

    latest_observation = (
        daily_5y
        .groupby(
            [
                "canonical_id",
                "series_type",
                "unit",
            ]
        )
        .agg(
            last_date=(
                "date",
                "max",
            ),
            last_value=(
                "close",
                "last",
            ),
        )
        .sort_values(
            "last_date"
        )
    )

    print()
    print("=" * 80)
    print("LATEST OBSERVATION")
    print("=" * 80)

    display(
        latest_observation
    )


# ============================================================
# US Rates / Inflation Check
#
# Nominal ≈ Real + Breakeven
# ============================================================

if not daily_5y.empty:

    rate_ids = [
        "US5Y_YIELD",
        "US5Y_REAL",
        "US5Y_BE",
        "US10Y_YIELD",
        "US10Y_REAL",
        "US10Y_BE",
    ]

    rate_check = (
        daily_5y.loc[
            daily_5y[
                "canonical_id"
            ].isin(
                rate_ids
            )
        ]
        .pivot_table(
            index="date",
            columns="canonical_id",
            values="close",
            aggfunc="last",
        )
        .sort_index()
    )

    required_5y = {
        "US5Y_YIELD",
        "US5Y_REAL",
        "US5Y_BE",
    }

    if required_5y.issubset(
        rate_check.columns
    ):

        rate_check[
            "US5Y_DECOMP_DIFF"
        ] = (
            rate_check[
                "US5Y_YIELD"
            ]
            - rate_check[
                "US5Y_REAL"
            ]
            - rate_check[
                "US5Y_BE"
            ]
        )

    required_10y = {
        "US10Y_YIELD",
        "US10Y_REAL",
        "US10Y_BE",
    }

    if required_10y.issubset(
        rate_check.columns
    ):

        rate_check[
            "US10Y_DECOMP_DIFF"
        ] = (
            rate_check[
                "US10Y_YIELD"
            ]
            - rate_check[
                "US10Y_REAL"
            ]
            - rate_check[
                "US10Y_BE"
            ]
        )

    print()
    print("=" * 80)
    print("US RATES / INFLATION CHECK")
    print("=" * 80)

    display(
        rate_check.tail(10)
    )


# ============================================================
# US Credit Check
# ============================================================

if not daily_5y.empty:

    credit_check = (
        daily_5y.loc[
            daily_5y[
                "canonical_id"
            ].isin(
                CREDIT_SPREAD_IDS
            )
        ]
        .pivot_table(
            index="date",
            columns="canonical_id",
            values="close",
            aggfunc="last",
        )
        .sort_index()
    )

    print()
    print("=" * 80)
    print("US CREDIT OAS CHECK")
    print("=" * 80)

    display(
        credit_check.tail(10)
    )


# ============================================================
# FX Options Check
# ============================================================

if not daily_5y.empty:

    fx_option_ids = (
        FX_VOL_IDS
        | FX_RR_IDS
    )

    fx_option_check = (
        daily_5y.loc[
            daily_5y[
                "canonical_id"
            ].isin(
                fx_option_ids
            )
        ]
        .pivot_table(
            index="date",
            columns="canonical_id",
            values="close",
            aggfunc="last",
        )
        .sort_index()
    )

    print()
    print("=" * 80)
    print("FX OPTIONS VOL / RISK REVERSAL CHECK")
    print("=" * 80)

    display(
        fx_option_check.tail(10)
    )


# ============================================================
# US Sector Total Return Check
# ============================================================

if not daily_5y.empty:

    us_sector_check = (
        daily_5y.loc[
            daily_5y[
                "canonical_id"
            ].isin(
                US_SECTOR_IDS
            )
        ]
        .pivot_table(
            index="date",
            columns="canonical_id",
            values="close",
            aggfunc="last",
        )
        .sort_index()
    )

    print()
    print("=" * 80)
    print("US SECTOR TOTAL RETURN CHECK")
    print("=" * 80)

    display(
        us_sector_check.tail(5)
    )


# ============================================================
# J-REIT Check
# ============================================================

if not daily_5y.empty:

    jreit_check = (
        daily_5y.loc[
            daily_5y[
                "canonical_id"
            ].eq(
                "JREIT"
            )
        ]
        .tail(10)
    )

    print()
    print("=" * 80)
    print("J-REIT TOTAL RETURN CHECK")
    print("=" * 80)

    display(
        jreit_check
    )


# ============================================================
# Output CSV
# ============================================================

if not daily_5y.empty:

    today_str = (
        datetime.now(
            ZoneInfo(TZ)
        )
        .strftime(
            "%Y%m%d"
        )
    )

    output_file = (
        OUTPUT_DIR
        / f"DCAM_Raw_5Y_Daily_{today_str}.csv"
    )

    daily_5y.to_csv(
        output_file,
        index=False,
        encoding="utf-8-sig",
    )

    file_size_mb = (
        output_file.stat().st_size
        / 1024
        / 1024
    )

    print()
    print("=" * 80)
    print("CSV OUTPUT")
    print("=" * 80)

    print(
        f"File : "
        f"{output_file}"
    )

    print(
        f"Size : "
        f"{file_size_mb:.2f} MB"
    )


# ============================================================
# Errors
# ============================================================

if not errors.empty:

    print()
    print("=" * 80)
    print("ERRORS")
    print("=" * 80)

    display(
        errors
    )


# ============================================================
# No Data
# ============================================================

if not no_data.empty:

    print()
    print("=" * 80)
    print("NO DATA")
    print("=" * 80)

    display(
        no_data
    )

DCAM Bloomberg Daily Download
Timezone    : Asia/Tokyo
Start date  : 2021-08-11
End date    : 2026-08-11
History     : 5 years
Instruments : 68

OK      | ES                       | 1,258 rows | 2021-08-11 -> 2026-08-11 | PX_LAST
OK      | NQ                       | 1,258 rows | 2021-08-11 -> 2026-08-11 | PX_LAST
OK      | RTY                      | 1,258 rows | 2021-08-11 -> 2026-08-11 | PX_LAST
OK      | TOPIX                    | 1,223 rows | 2021-08-11 -> 2026-08-10 | PX_LAST
OK      | JREIT                    | 1,223 rows | 2021-08-11 -> 2026-08-10 | PX_LAST
OK      | EURO                     | 1,275 rows | 2021-08-11 -> 2026-08-11 | PX_LAST
OK      | DAX                      | 1,275 rows | 2021-08-11 -> 2026-08-11 | PX_LAST
OK      | TAIWAN                   | 1,301 rows | 2021-08-11 -> 2026-08-11 | PX_LAST
OK      | KOSPI                    | 1,222 rows | 2021-08-11 -> 2026-08-11 | PX_LAST
OK      | CHINA_A50                | 1,301 rows | 2021-08-11 -> 2026-08-11 | PX_LAST
OK   

,,,,,,rows,first_date,last_date,first_value,last_value
canonical_id,asset_class,region,series_type,unit,source_field,,,,,
AUDUSD,FX,GLOBAL,PRICE,PRICE,PX_LAST,1305,2021-08-11,2026-08-11,0.7374,0.7061
BOBL,RATES,EUROPE,PRICE,PRICE,PX_LAST,1275,2021-08-11,2026-08-11,133.7290,114.0400
BRENT,ENERGY,GLOBAL,PRICE,PRICE,PX_LAST,1293,2021-08-11,2026-08-11,71.4400,88.4400
BTP,RATES,EUROPE,PRICE,PRICE,PX_LAST,1275,2021-08-11,2026-08-11,135.9200,116.7100
BUND,RATES,EUROPE,PRICE,PRICE,PX_LAST,1275,2021-08-11,2026-08-11,164.9200,124.7300
...,...,...,...,...,...,...,...,...,...,...
US_UTILITIES,EQUITY_SECTOR,US,TOTAL_RETURN_INDEX,INDEX,TOT_RETURN_INDEX_GROSS_DVDS,1255,2021-08-11,2026-08-11,34.0650,50.5212
VIX,VOLATILITY,US,VOLATILITY,PRICE,PX_LAST,1287,2021-08-11,2026-08-11,16.0600,15.4500
VIX_F1,VOLATILITY,US,VOLATILITY,PRICE,PX_LAST,1258,2021-08-11,2026-08-11,19.7318,18.5500



DATA QUALITY
Duplicate canonical_id/date : 0
Missing close               : 0
Invalid positive levels <=0 : 0

LATEST OBSERVATION


,,,last_date,last_value
canonical_id,series_type,unit,,
US_IG_OAS,CREDIT_SPREAD,PCT,2026-08-10,0.7700
TOPIX,PRICE,PRICE,2026-08-10,4116.5000
JGB,PRICE,PRICE,2026-08-10,126.9000
JREIT,TOTAL_RETURN_INDEX,INDEX,2026-08-10,4949.8200
MOVE,VOLATILITY,PRICE,2026-08-10,75.4600
...,...,...,...,...
SOYBEAN,PRICE,PRICE,2026-08-11,1167.7500
TAIWAN,PRICE,PRICE,2026-08-11,3903.2500
WHEAT,PRICE,PRICE,2026-08-11,648.5000



US RATES / INFLATION CHECK


...


US SECTOR TOTAL RETURN CHECK


canonical_id,US_COMM_SERVICES,US_CONSUMER_DISC,US_CONSUMER_STAPLES,US_ENERGY,US_FINANCIALS,US_HEALTHCARE,US_INDUSTRIALS,US_MATERIALS,US_REAL_ESTATE,US_TECH,US_UTILITIES
date,,,,,,,,,,,
2026-08-05,116.7097,123.6041,97.3406,68.6769,63.2324,177.9741,200.9086,58.1043,53.4045,193.0026,50.7145
2026-08-06,117.0360,123.0415,97.0896,69.6955,63.0252,178.2885,199.1944,57.5855,52.9437,192.4005,50.3892
2026-08-07,117.1097,124.8751,97.1011,68.9046,62.7963,179.6220,199.6472,58.3471,53.1446,195.1412,50.6564
2026-08-10,117.7202,124.6772,96.9071,72.1161,63.0252,182.6143,199.0219,58.7003,52.4593,193.4282,50.0988
2026-08-11,117.7203,124.3646,96.5706,72.8651,63.2051,182.0071,200.4127,58.6893,52.3411,193.1687,50.5212



J-REIT TOTAL RETURN CHECK


,date,canonical_id,asset_class,region,series_type,unit,close,source_ticker,source_field
86003,2026-07-28,JREIT,REIT,JAPAN,TOTAL_RETURN_INDEX,INDEX,5150.36,TPXDREIT Index,PX_LAST
86071,2026-07-29,JREIT,REIT,JAPAN,TOTAL_RETURN_INDEX,INDEX,5229.38,TPXDREIT Index,PX_LAST
86139,2026-07-30,JREIT,REIT,JAPAN,TOTAL_RETURN_INDEX,INDEX,5192.50,TPXDREIT Index,PX_LAST
86207,2026-07-31,JREIT,REIT,JAPAN,TOTAL_RETURN_INDEX,INDEX,5083.42,TPXDREIT Index,PX_LAST
86275,2026-08-03,JREIT,REIT,JAPAN,TOTAL_RETURN_INDEX,INDEX,4992.67,TPXDREIT Index,PX_LAST
86343,2026-08-04,JREIT,REIT,JAPAN,TOTAL_RETURN_INDEX,INDEX,4951.48,TPXDREIT Index,PX_LAST
86411,2026-08-05,JREIT,REIT,JAPAN,TOTAL_RETURN_INDEX,INDEX,4960.15,TPXDREIT Index,PX_LAST
86479,2026-08-06,JREIT,REIT,JAPAN,TOTAL_RETURN_INDEX,INDEX,4964.92,TPXDREIT Index,PX_LAST
86547,2026-08-07,JREIT,REIT,JAPAN,TOTAL_RETURN_INDEX,INDEX,4952.88,TPXDREIT Index,PX_LAST
86615,2026-08-10,JREIT,REIT,JAPAN,TOTAL_RETURN_INDEX,INDEX,4949.82,TPXDREIT Index,PX_LAST



CSV OUTPUT
File : C:\Data\DCAM\DCAM_Raw_5Y_Daily_20260812.csv
Size : 7.28 MB
